# 课程 02 - 探索 Microsoft Agent 框架

**Microsoft Agent 框架（MAF）** 是一个用于构建 AI 代理的统一框架。它提供了一个简洁、可组合的架构，包含四个核心构建模块：

- <strong>客户端</strong> – 连接到 AI 模型端点并处理通信
- <strong>代理</strong> – 包装客户端，带有指令和工具定义
- <strong>工具</strong> – 通过模型可调用的自定义函数扩展代理能力
- <strong>会话</strong> – 维护多轮交互的对话历史

在本课中，我们将构建一个使用这些概念来检查目的地可用性的<strong>旅行预订代理</strong>。


## 设置


In [1]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.openai import OpenAIChatCompletionClient

dotenv.load_dotenv(dotenv.find_dotenv())

True

## 理解 Agent 框架架构

Microsoft Agent 框架遵循分层架构：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. <strong>客户端</strong> – `FoundryChatClient` 连接到 Azure OpenAI 部署，处理身份验证、请求格式化和响应解析。
2. **Agent** – 通过 `provider.create_agent()` 从客户端创建，agent 结合了模型访问、指令（系统提示）和工具。
3. <strong>工具</strong> – 用 `@tool` 装饰的 Python 函数，agent 可以调用它们执行操作或获取数据。
4. <strong>会话</strong> – `AgentSession` 对象（通过 `agent.create_session()` 创建），存储对话历史，实现多轮对话，agent 记忆之前的上下文。

让我们一步步构建每一层。


In [9]:
import os
# 创建客户端——这是与 AI 模型的连接
endpoint = os.getenv("LLM_BASE_URL")
model = os.getenv("LLM_MODEL")

if not endpoint or not model:
    raise ValueError(
        "缺少必需的环境变量。请设置 LLM_BASE_URL 和 LLM_MODEL 环境变量"
        "（例如在 .env 文件或 shell 环境中）。"
    )

provider = OpenAIChatCompletionClient(
    model=os.environ["LLM_MODEL"],
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
)

## 使用 @tool 装饰器添加工具

工具让代理可以执行生成文本以外的操作。`@tool` 装饰器将普通的 Python 函数转换为代理可以调用的功能。

关键点：
- 使用 `Annotated[type, "description"]`，让模型理解每个参数。
- 文档字符串变成模型看到的工具描述。
- `approval_mode="never_require"` 表示工具自动运行，无需用户确认。


In [3]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "要查询可用性的目的地"]
) -> str:
    """检查某个度假目的地当前是否可预订。"""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} {'可预订' if is_available else '不可预订'}。"

## 使用工具创建代理

现在我们将客户端、指令和工具组合成一个代理。`instructions` 作为系统提示——它们定义了代理的角色和行为。


In [11]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "你是一个旅行预订代理。帮助用户查询目的地可用性并给出推荐。"
        "在推荐任何目的地之前，务必先检查其可用性。"
    ),
    tools=[check_destination_availability],
)

## 多轮对话会话

`AgentSession`（通过 `agent.create_session()` 创建）跟踪对话中的所有消息。通过在每次 `agent.run()` 调用中传递相同的会话，代理可以访问完整的对话历史并引用之前的消息。

我们传入 `tools=[check_destination_availability]`，以便代理在每轮中都能调用我们的可用性检查器。


In [7]:
session = agent.create_session()

# 第一轮：询问有哪些可用目的地
response = await agent.run(
    "你有哪些目的地可以预订？",
    session=session,
)
print(f"智能体：{response}")

Agent: I can check availability for specific destinations, but I don't have a way to browse all available options in our system. 

Could you tell me which destinations you're interested in? I can check their availability right away. For example, are you looking at beach resorts, cities, mountain getaways, or specific places like Paris, Tokyo, Bali, or the Maldives?


In [12]:
# 第二轮：追问——agent 记得之前的对话
response = await agent.run(
    "我想去一个温暖的地方。有哪些可以预订的？",
    session=session,
)
print(f"智能体：{response}")

Agent: Let me check availability for some popular warm destinations for you!I checked several popular warm destinations for you, but unfortunately, the following are all currently **not available** for booking:

- Bali
- Maldives
- Cancun
- Hawaii
- Phuket
- Costa Rica
- Miami
- Dubai
- Caribbean
- Santorini

This could be due to high demand or limited inventory. Do you have any other warm destinations in mind? I can check places like:

- **Mexico** (Tulum, Puerto Vallarta, Cabo San Lucas)
- **Southeast Asia** (Vietnam, Philippines)
- **South America** (Brazil, Colombia)
- **Africa/Middle East** (Morocco, Egypt, Oman)
- **Southern Europe** (Spain, Portugal, Italy)

Just let me know a few places you're interested in and I'll check their availability right away!


## 总结

在本课中，您探索了微软代理框架的四大支柱：

| 概念 | 你学到了什么 |
|---------|------------------|
| <strong>客户端</strong> | `FoundryChatClient` 使用基于凭证的认证连接到 Azure OpenAI |
| <strong>代理</strong> | `provider.create_agent()` 将模型连接与指令和名称绑定在一起 |
| <strong>工具</strong> | `@tool` 装饰器暴露 Python 函数供代理调用 |
| <strong>会话</strong> | `agent.create_session()` 跨多轮保持对话历史 |

这些构建模块组合在一起，创建能够进行自然对话、调用外部函数并保持上下文的代理 —— 这是后续课程中更高级代理模式的基础。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
